# Fine-tune Qwen3-0.6B with LoRA for receipt and bill field extraction

Run in Google Colab with a GPU (L4 or T4). Upload `train.jsonl` and `test.jsonl` (made by `ml/build_dataset.py`) through the Files panel first. They hold real financial data, so never commit them or this notebook's outputs.

## 1. Install

In [ ]:
!pip -q install -U transformers peft accelerate
!pip -q uninstall -y torchao  # an old torchao on Colab breaks peft
DATA = "/content"
import os; print(os.listdir(DATA))

## 2. Prompt (must match `ml/prompt.py` exactly)

In [ ]:
SYSTEM = """You extract fields from OCR text of Canadian receipts, bills and invoices.
Return only a JSON object with exactly these keys:
doc_type, vendor, gst_number, invoice_number, date, due_date, subtotal, fees, discounts,
currency, gst_hst, pst, tax_total, total.
Rules: doc_type is one of bill, receipt, invoice, statement, payment (from the business's side:
bill = asks us to pay, receipt = proof we paid). Use null for anything not printed; never guess
or calculate. Amounts are plain numbers. Dates are YYYY-MM-DD. vendor is a short brand name.
gst_number has no spaces. Prefer the invoice/receipt number over order, PO or booking numbers.
A single unlabelled tax line goes in tax_total. Discounts are positive numbers."""

## 3. Load Qwen and add LoRA

In [ ]:
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).cuda()
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

## 4. Tokenize (loss only on the JSON answer, thinking off)

In [ ]:
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
train, test = load(f"{DATA}/train.jsonl"), load(f"{DATA}/test.jsonl")

def prompt_ids(row):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": row["input"]}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return tok(text, add_special_tokens=False)["input_ids"]

def encode(row):
    p = prompt_ids(row)
    a = tok(json.dumps(row["output"], ensure_ascii=False) + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

train_ds = [encode(r) for r in train]
print(len(train_ds), "examples, longest:", max(len(x["input_ids"]) for x in train_ds), "tokens")

def collate(batch):
    n, pad = max(len(b["input_ids"]) for b in batch), tok.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch]),
    }

## 5. Train

In [ ]:
args = TrainingArguments(
    output_dir="/content/out", num_train_epochs=5, per_device_train_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=1, logging_steps=2, save_strategy="no", bf16=True,
    report_to="none", remove_unused_columns=False)
Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate).train()

## 6. Predict on the test set and download the predictions
Score locally with `python ml/score.py data/test.jsonl data/pred_qwen_lora.jsonl`.

In [ ]:
model.eval()
with open(f"{DATA}/pred_qwen_lora.jsonl", "w", encoding="utf-8") as out:
    for row in test:
        ids = torch.tensor([prompt_ids(row)]).cuda()
        with torch.no_grad():
            gen = model.generate(ids, max_new_tokens=300, do_sample=False)
        text = tok.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
        try: pred = json.loads(text)
        except json.JSONDecodeError: pred = {}
        out.write(json.dumps({"file": row["file"], "prediction": pred, "raw": text}, ensure_ascii=False) + "\n")
        print(row["file"], text[:120])

from google.colab import files
files.download(f"{DATA}/pred_qwen_lora.jsonl")

## 7. Export: save the adapter, merge, convert to GGUF for Ollama
Download `fintrace-qwen3-0.6b.gguf` and `lora-adapter.zip` from the Files panel before the session ends.

In [ ]:
import json
model.save_pretrained("/content/lora-adapter")
merged = model.merge_and_unload()
merged.save_pretrained("/content/merged"); tok.save_pretrained("/content/merged")

# llama.cpp's converter uses an older transformers that can't read this newer key
p = "/content/merged/tokenizer_config.json"
cfg = json.load(open(p))
if isinstance(cfg.get("extra_special_tokens"), list):
    cfg.pop("extra_special_tokens")
json.dump(cfg, open(p, "w"), indent=2)

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/fintrace-qwen3-0.6b.gguf --outtype q8_0
!cd /content && zip -qr lora-adapter.zip lora-adapter
!ls -lh /content/*.gguf /content/*.zip